# Medical Form Understanding & Population Pipeline

This notebook implements an AI-assisted document understanding pipeline for medical forms.

## Overview
1. **Schema Extraction** - Extract form field schema from the fillable PDF (`schema.json`)
2. **Information Extraction** - Extract patient information from documents and produce `answers.json`
3. **PDF Population (optional)** - Populate `form_fillable.pdf` into `populated.pdf` (text fields only)
4. **Evaluation** - Run a small, reliability-focused mini-evaluation

**Outputs generated by this notebook**:
- `schema.json`
- `answers.json`
- `populated.pdf`

---

## Task 1: Schema Extraction

Extract field schemas from the fillable PDF, including labels, types, bounding boxes, and normalized names.

In [1]:
# Import schema extraction module
import sys
sys.path.append('src')

from schema_extraction import SchemaExtractor, extract_schema

# Extract schema from fillable PDF
pdf_path = "inputs/form_fillable.pdf"
schema = extract_schema(pdf_path, output_path="outputs/schema.json", filter_clear_button=True)

print(f"✅ Extracted schema with {len(schema)} fields")
print("\nSample fields:")
for i, (key, field) in enumerate(list(schema.items())[:5]):
    print(f"\n{key}:")
    print(f"  Label: {field['label']}")
    print(f"  PDF Name: {field['pdf_name']}")
    print(f"  Type: {field['type']}")
    if field.get('group'):
        print(f"  Group: {field['group']}, Part: {field['part']}")

✅ Extracted schema with 49 fields

Sample fields:

home_phone_area_code:
  Label: Home Phone # (+ Area Code)
  PDF Name: APS1. areacode
  Type: text
  Group: home_phone, Part: area_code

address_street_city_province_postal_code:
  Label: Address (Street, City, Province, Postal Code)
  PDF Name: APS1.Address
  Type: text

certificate_if_applicable:
  Label: Certificate # (if applicable)
  PDF Name: APS1.Cert
  Type: text

contract_or_policy:
  Label: Contract or Policy #
  PDF Name: APS1.Contract
  Type: text

date_last_day:
  Label: Date Last Worked (dd)
  PDF Name: APS1.Date_last_d
  Type: text
  Group: date_last, Part: day


## Task 2: Information Extraction

Extract patient information from the provided patient documents:
- `demographics.json` (structured, treated as the anchor source)
- `soap_notes.txt` (noisy clinical note; evidence-gated extraction)
- `lab_result.pdf` (generic document; embedded-text first with optional OCR fallback)

**Implementation**: `src/information_extraction.py` (`InformationExtractor`).


In [2]:
# Extract information from demographics.json using the module
import sys
sys.path.append('src')

from information_extraction import InformationExtractor
import json

# Load schema
with open('outputs/schema.json', 'r') as f:
    schema = json.load(f)

# Create extractor
extractor = InformationExtractor(schema)

# Extract from demographics.json
print("="*60)
print("EXTRACTING FROM DEMOGRAPHICS.JSON")
print("="*60)

results = extractor.extract_from_demographics('inputs/demographics.json', confidence=0.95)

print(f"\n✅ Extracted {len(results)} fields from demographics.json")
print("\n📊 Extracted Fields:")
for field_name, field_data in sorted(results.items()):
    print(f"\n  {field_name}:")
    print(f"    Value: {field_data['value']}")
    print(f"    Source: {field_data['source']}")
    print(f"    Confidence: {field_data['confidence']}")
    print(f"    Transformation: {field_data['transformation']}")

# Show phone parsing verification
print("\n" + "="*60)
print("PHONE PARSING VERIFICATION")
print("="*60)

phone_groups = {
    "home_phone": ["home_phone_area_code", "home_phone_first3", "home_phone_last4"],
    "cell_phone": ["cell_phone_area_code", "cell_phone_first3", "cell_phone_last4"]
}

for phone_type, field_list in phone_groups.items():
    print(f"\n{phone_type.upper().replace('_', ' ')}:")
    all_present = all(field in results for field in field_list)
    if all_present:
        area = results[field_list[0]]['value']
        first3 = results[field_list[1]]['value']
        last4 = results[field_list[2]]['value']
        print(f"  ✓ Area Code: {area}")
        print(f"  ✓ First 3: {first3}")
        print(f"  ✓ Last 4: {last4}")
        print(f"  → Combined: {area}-{first3}-{last4}")
    else:
        print(f"  ✗ Missing fields")

EXTRACTING FROM DEMOGRAPHICS.JSON

✅ Extracted 11 fields from demographics.json

📊 Extracted Fields:

  address_street_city_province_postal_code:
    Value: 45 Maple Ave, Toronto, ON, K7L 3V8
    Source: demographics.json
    Confidence: 0.95
    Transformation: address_format

  cell_phone_area_code:
    Value: 647
    Source: demographics.json
    Confidence: 0.95
    Transformation: phone_parse.area_code

  cell_phone_first3:
    Value: 666
    Source: demographics.json
    Confidence: 0.95
    Transformation: phone_parse.first3

  cell_phone_last4:
    Value: 8888
    Source: demographics.json
    Confidence: 0.95
    Transformation: phone_parse.last4

  date_of_birth_dd:
    Value: 15
    Source: demographics.json
    Confidence: 0.95
    Transformation: date_parse.day

  date_of_birth_mm:
    Value: 04
    Source: demographics.json
    Confidence: 0.95
    Transformation: date_parse.month

  date_of_birth_yyyy:
    Value: 1960
    Source: demographics.json
    Confidence: 0.95
  

In [3]:
# ============================================================================
# SOAP extraction (in src/), completion-only
# ============================================================================
# - Demographics is the anchor truth for identity/contact fields.
# - SOAP is used to fill missing clinical fields (diagnoses, explicit meds, etc.).
# - Deterministic-first, evidence-gated; optional LLM fallback for remaining missing keys.

# (Re-)load SOAP text for downstream evaluation cell
with open('inputs/soap_notes.txt', 'r') as f:
    soap_text = f.read()

soap_results = extractor.extract_from_soap_notes(
    'inputs/soap_notes.txt',
    already_filled=results,
    use_llm_fallback=True,
    model_name='Qwen/Qwen2.5-3B-Instruct',
    model_cache_dir='./models/Qwen2.5-3B-Instruct',
)

kept = {k: v for k, v in soap_results.items() if v.get('value') is not None}
print('=' * 60)
print('SOAP EXTRACTION (src/information_extraction.py)')
print('=' * 60)
print(f"✅ Kept fields: {len(kept)}")
for k, v in kept.items():
    print(f"\n{k}:")
    print(f"  value: {v['value']}")
    print(f"  confidence: {v['confidence']}")
    print(f"  evidence: {v['evidence']}")
    print(f"  reasoning: {v['reasoning']}")


The tokenizer you are loading from 'models/Qwen2.5-3B-Instruct' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


SOAP EXTRACTION (src/information_extraction.py)
✅ Kept fields: 6

primary_1:
  value: stable angina given exertional pattern and resolution w/ rest
  confidence: 0.75
  evidence: Likely stable angina given exertional pattern and resolution w/ rest. Multiple risk factors: HTN, age, family hx (father MI at 58).
  reasoning: extracted from Assessment section

primary_2:
  value: Hypertension
  confidence: 0.75
  evidence: Hypertension — borderline control; adherence inconsistent.
  reasoning: extracted from Assessment section

secondary_and_or_complications_1:
  value: GERD
  confidence: 0.75
  evidence: GERD — chronic but unlikely to explain exertional symptoms.
  reasoning: extracted from Assessment section

secondary_and_or_complications_2:
  value: Hyperlipidemia
  confidence: 0.75
  evidence: Hyperlipidemia — labs overdue.
  reasoning: extracted from Assessment section

medication_1_name:
  value: nitroglycerin
  confidence: 0.8
  evidence: Provide nitroglycerin SL for exertional dis

In [4]:
# ============================================================================
# Reconciliation: demographics (anchor) + SOAP (fill missing / flag conflicts)
# ============================================================================
# Assumptions:
# - `results` exists from the demographics extraction cell.
# - `soap_results` and `soap_text` exist from the SOAP extraction cell.

import re


def _norm(s):
    if s is None:
        return None
    return re.sub(r"\s+", " ", str(s)).strip().lower()

combined_results = dict(results)  # start with high-confidence structured values
conflicts = []
verified = []
added_from_soap = []

for k, soap_obj in soap_results.items():
    soap_val = soap_obj.get("value")
    if soap_val is None:
        continue

    if k not in combined_results:
        combined_results[k] = soap_obj
        added_from_soap.append(k)
        continue

    demo_val = combined_results[k].get("value")
    if _norm(demo_val) == _norm(soap_val):
        # Keep demographics, just record that SOAP corroborated it
        combined_results[k] = {
            **combined_results[k],
            "verified_by": "inputs/soap_notes.txt",
            "verification_evidence": soap_obj.get("evidence"),
        }
        verified.append(k)
    else:
        # Do not overwrite; flag for human review
        conflicts.append({
            "field": k,
            "demographics_value": demo_val,
            "soap_value": soap_val,
            "soap_evidence": soap_obj.get("evidence"),
        })
        combined_results[k] = {
            **combined_results[k],
            "needs_review": True,
            "conflict_with": "inputs/soap_notes.txt",
            "conflict_candidate": soap_val,
            "conflict_evidence": soap_obj.get("evidence"),
        }

print("=" * 60)
print("RECONCILIATION SUMMARY")
print("=" * 60)
print(f"Added from SOAP: {len(added_from_soap)} -> {added_from_soap}")
print(f"Verified by SOAP: {len(verified)} -> {verified}")
print(f"Conflicts flagged: {len(conflicts)}")

if conflicts:
    print("\n⚠️  Conflicts (kept demographics; marked needs_review):")
    for c in conflicts:
        print(f"\n- {c['field']}")
        print(f"  demographics: {c['demographics_value']}")
        print(f"  soap: {c['soap_value']}")
        print(f"  evidence: {c['soap_evidence']}")

print("\n✅ Combined extracted fields:", len(combined_results))

# ============================================================================
# Mini-eval (hallucination-focused)
# ============================================================================
# 1) Evidence match rate on SOAP-kept fields
# 2) Dose hallucination trap: dose fields should be null unless explicitly numeric in note

soap_kept = {k: v for k, v in soap_results.items() if v.get("value") is not None}
soap_evidence_ok = 0

soap_text_norm = re.sub(r"\s+", " ", soap_text).strip()
for k, v in soap_kept.items():
    ev = v.get("evidence")
    if isinstance(ev, str) and re.sub(r"\s+", " ", ev).strip() in soap_text_norm:
        soap_evidence_ok += 1

print("\n" + "=" * 60)
print("SOAP MINI-EVAL")
print("=" * 60)
print(f"SOAP kept fields: {len(soap_kept)}")
print(f"Evidence match rate (kept): {soap_evidence_ok}/{len(soap_kept)}" if soap_kept else "Evidence match rate (kept): n/a")

bad_doses = []
for k, v in soap_kept.items():
    if k.endswith("_dose"):
        if not re.search(r"\d", str(v.get("value"))):
            bad_doses.append(k)

print(f"Dose hallucination checks (should be empty): {bad_doses}")


RECONCILIATION SUMMARY
Added from SOAP: 6 -> ['primary_1', 'primary_2', 'secondary_and_or_complications_1', 'secondary_and_or_complications_2', 'medication_1_name', 'medication_1_often']
Verified by SOAP: 0 -> []
Conflicts flagged: 0

✅ Combined extracted fields: 17

SOAP MINI-EVAL
SOAP kept fields: 6
Evidence match rate (kept): 6/6
Dose hallucination checks (should be empty): []


In [5]:
# ============================================================================
# Lab result extraction (generic, optional): fill only remaining missing fields
# ============================================================================
# Per problem.md: lab_result.pdf can contain anything, so we keep this conservative and generic.
# - Prefer embedded PDF text extraction; OCR is only used as a fallback if the PDF has no text.
# - Only fill fields that are still missing after demographics + SOAP.

lab_results = extractor.extract_from_lab_result(
    'inputs/lab_result.pdf',
    already_filled=combined_results,
    use_ocr_fallback=True,
)

added = sorted([k for k, v in lab_results.items() if v.get('value') is not None])
print('=' * 60)
print('LAB RESULT EXTRACTION')
print('=' * 60)
print(f"✅ Added fields from lab_result.pdf: {len(added)}")
print('   ', added)

# Merge into combined results
combined_results = extractor.combine_extractions(combined_results, lab_results)
print(f"✅ Combined extracted fields after lab_result.pdf: {len(combined_results)}")


LAB RESULT EXTRACTION
✅ Added fields from lab_result.pdf: 7
    ['medication_1_dose', 'medication_2_dose', 'medication_2_name', 'medication_2_often', 'medication_3_dose', 'medication_3_name', 'medication_3_often']
✅ Combined extracted fields after lab_result.pdf: 24


In [6]:
# ============================================================================
# Generate answers.json (deliverable)
# ============================================================================
# Format required by problem.md: { field_key: { value, source, reasoning, confidence } }

import json

answers = {}
for field_key, obj in combined_results.items():
    if not isinstance(obj, dict):
        continue
    if obj.get('value') is None or str(obj.get('value')).strip() == '':
        continue

    answers[field_key] = {
        'value': obj.get('value'),
        'source': obj.get('source'),
        'reasoning': obj.get('reasoning') or obj.get('transformation') or '',
        'confidence': obj.get('confidence', None),
    }

with open('outputs/answers.json', 'w') as f:
    json.dump(answers, f, indent=2)

print(f"✅ Wrote answers.json with {len(answers)} populated fields")
print("Sample:")
for k in list(answers.keys())[:8]:
    print(f"- {k}: {answers[k]['value']} ({answers[k]['source']})")


✅ Wrote answers.json with 24 populated fields
Sample:
- patient_name_last_first_middle_initial: Peter Julius Fern (demographics.json)
- date_of_birth_dd: 15 (demographics.json)
- date_of_birth_mm: 04 (demographics.json)
- date_of_birth_yyyy: 1960 (demographics.json)
- home_phone_area_code: 613 (demographics.json)
- home_phone_first3: 656 (demographics.json)
- home_phone_last4: 5890 (demographics.json)
- cell_phone_area_code: 647 (demographics.json)


## Task 4: PDF Population (fillable PDF)

Populate `form_fillable.pdf` using `answers.json`.

- **Scope**: text fields only (skip checkboxes/radios/choice fields)
- **Keying**: uses `schema.json` → `pdf_name` (AcroForm field name) to avoid label ambiguity

**Implementation**: `src/pdf_population.py` (`populate_fillable_pdf`).


In [7]:
# ============================================================================
# Task 4 (Optional): PDF Population (fillable PDF, text fields only)
# ============================================================================

import sys
sys.path.append('src')

from pdf_population import populate_fillable_pdf

out_path, report = populate_fillable_pdf(
    form_pdf_path='inputs/form_fillable.pdf',
    schema_path='outputs/schema.json',
    answers_path='outputs/answers.json',
    output_path='outputs/populated.pdf',
)

print('=' * 60)
print('PDF POPULATION SUMMARY')
print('=' * 60)
print(f"✅ Wrote: {out_path}")
print(f"✅ Filled text fields: {len(report.filled)}")
print(f"⚠️  Skipped: {len(report.skipped)}")
print(f"⚠️  Missing PDF fields: {len(report.missing_pdf_fields)}")

# Show a small sample
for k in list(report.filled.keys())[:12]:
    print(f"- {k}: {report.filled[k]}")

if report.missing_pdf_fields:
    print("\nMissing PDF fields (schema key -> pdf_name):")
    for k, v in list(report.missing_pdf_fields.items())[:12]:
        print(f"- {k} -> {v}")


PDF POPULATION SUMMARY
✅ Wrote: populated.pdf
✅ Filled text fields: 24
⚠️  Skipped: 0
⚠️  Missing PDF fields: 0
- patient_name_last_first_middle_initial: Peter Julius Fern
- date_of_birth_dd: 15
- date_of_birth_mm: 04
- date_of_birth_yyyy: 1960
- home_phone_area_code: 613
- home_phone_first3: 656
- home_phone_last4: 5890
- cell_phone_area_code: 647
- cell_phone_first3: 666
- cell_phone_last4: 8888
- address_street_city_province_postal_code: 45 Maple Ave, Toronto, ON, K7L 3V8
- primary_1: stable angina given exertional pattern and resolution w/ rest


## Task 5: Evaluation (mini evaluation)

A small, reliability-focused evaluation aligned with `problem.md`:
- formatting checks (DOB / phone parts)
- hallucination guards for SOAP extraction (evidence match, no invented dose)
- lab medication sanity + dedup checks
- populated PDF readback (verify filled AcroForm values exist)

**Implementation**: executed in-notebook in the Task 5 code cell.


In [8]:
# ============================================================================
# Task 5: Mini Evaluation Component (Required)
# ============================================================================
# Goal (per problem.md): demonstrate sound engineering instincts.
# We evaluate a few reliability-focused invariants:
# - Formatting consistency (DOB / phone parts)
# - SOAP hallucination guards (evidence-backed, no invented dose)
# - Lab medication sanity (dose numeric, frequency recognized)
# - Dedup check (no duplicate medication names across slots)
# - PDF population verification (AcroForm values exist in populated.pdf)

import json
import re
from pypdf import PdfReader


def _is_digits(s: str, n: int) -> bool:
    return isinstance(s, str) and len(s) == n and s.isdigit()


def _norm_med(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()


print('=' * 60)
print('EVALUATION SUMMARY')
print('=' * 60)

# -----------------------------
# 1) Formatting consistency checks (answers.json)
# -----------------------------
with open('outputs/answers.json', 'r') as f:
    answers = json.load(f)

fmt_checks = {
    'dob_dd': _is_digits(str(answers.get('date_of_birth_dd', {}).get('value', '')), 2),
    'dob_mm': _is_digits(str(answers.get('date_of_birth_mm', {}).get('value', '')), 2),
    'dob_yyyy': _is_digits(str(answers.get('date_of_birth_yyyy', {}).get('value', '')), 4),
    'home_phone_area_code': _is_digits(str(answers.get('home_phone_area_code', {}).get('value', '')), 3),
    'home_phone_first3': _is_digits(str(answers.get('home_phone_first3', {}).get('value', '')), 3),
    'home_phone_last4': _is_digits(str(answers.get('home_phone_last4', {}).get('value', '')), 4),
    'cell_phone_area_code': _is_digits(str(answers.get('cell_phone_area_code', {}).get('value', '')), 3),
    'cell_phone_first3': _is_digits(str(answers.get('cell_phone_first3', {}).get('value', '')), 3),
    'cell_phone_last4': _is_digits(str(answers.get('cell_phone_last4', {}).get('value', '')), 4),
}
passed = sum(1 for v in fmt_checks.values() if v)
print(f"Formatting checks passed: {passed}/{len(fmt_checks)}")
failed = [k for k, v in fmt_checks.items() if not v]
if failed:
    print('Failed formatting checks:', failed)

# -----------------------------
# 2) SOAP hallucination checks
# -----------------------------
# Assumes soap_results and soap_text exist from earlier cells.
soap_kept = {k: v for k, v in soap_results.items() if v.get('value') is not None}
soap_text_norm = re.sub(r"\s+", " ", soap_text).strip()

evidence_ok = 0
for k, v in soap_kept.items():
    ev = v.get('evidence')
    if isinstance(ev, str) and re.sub(r"\s+", " ", ev).strip() in soap_text_norm:
        evidence_ok += 1

bad_dose = []
for k, v in soap_kept.items():
    if k.endswith('_dose') and v.get('value') is not None:
        if not re.search(r"\d", str(v.get('value'))):
            bad_dose.append(k)

print(f"SOAP evidence match rate: {evidence_ok}/{len(soap_kept)}" if soap_kept else "SOAP evidence match rate: n/a")
print('SOAP invented-dose fields (should be empty):', bad_dose)

# -----------------------------
# 3) Lab medication sanity checks
# -----------------------------
# Assumes lab_results exists from earlier cells.
allowed_freq = {
    'once a day', 'twice a day', 'three times a day',
    'once daily', 'twice daily',
    'daily', 'every day',
    'bid', 'tid', 'qid',
    'prn', 'as needed'
}

lab_meds = []
for i in range(1, 6):
    name = answers.get(f'medication_{i}_name', {}).get('value')
    dose = answers.get(f'medication_{i}_dose', {}).get('value')
    often = answers.get(f'medication_{i}_often', {}).get('value')
    if name:
        lab_meds.append((i, name, dose, often))

lab_dose_ok = 0
lab_often_ok = 0
for i, name, dose, often in lab_meds:
    if dose is None or re.fullmatch(r"\d+(?:\.\d+)?", str(dose).strip()):
        lab_dose_ok += 1
    if often is None or str(often).strip().lower() in allowed_freq:
        lab_often_ok += 1

print(f"Medication dose numeric sanity: {lab_dose_ok}/{len(lab_meds)}" if lab_meds else "Medication dose numeric sanity: n/a")
print(f"Medication frequency whitelist sanity: {lab_often_ok}/{len(lab_meds)}" if lab_meds else "Medication frequency whitelist sanity: n/a")

# Dedup check (normalized med names)
seen = {}
dupes = []
for i, name, _, _ in lab_meds:
    n = _norm_med(name)
    if n in seen:
        dupes.append((seen[n], i, name))
    else:
        seen[n] = i
print('Duplicate medication names across slots (should be empty):', dupes)

# -----------------------------
# 4) populated.pdf verification
# -----------------------------
r = PdfReader('outputs/populated.pdf')
fields = r.get_fields() or {}
# Verify a few key fields are set
check_pdf = {
    'APS1.First Name': fields.get('APS1.First Name', {}).get('/V'),
    'APS1.Address': fields.get('APS1.Address', {}).get('/V'),
    'APS1.Date_of_birth_d': fields.get('APS1.Date_of_birth_d', {}).get('/V'),
    'APS1.Medication1': fields.get('APS1.Medication1', {}).get('/V'),
    'APS1.Dose1': fields.get('APS1.Dose1', {}).get('/V'),
}
non_empty = {k: v for k, v in check_pdf.items() if v not in (None, '')}
print(f"populated.pdf field readback (non-empty): {len(non_empty)}/{len(check_pdf)}")
for k, v in non_empty.items():
    print(f"- {k}: {v}")


EVALUATION SUMMARY
Formatting checks passed: 9/9
SOAP evidence match rate: 6/6
SOAP invented-dose fields (should be empty): []
Medication dose numeric sanity: 3/3
Medication frequency whitelist sanity: 2/3
Duplicate medication names across slots (should be empty): []
populated.pdf field readback (non-empty): 5/5
- APS1.First Name: Peter Julius Fern
- APS1.Address: 45 Maple Ave, Toronto, ON, K7L 3V8
- APS1.Date_of_birth_d: 15
- APS1.Medication1: nitroglycerin
- APS1.Dose1: 0.4
